# 2. MODEL TRAINING & EXPERIMENTATION
## Hospital Readmission Risk Predictor — UCI Diabetes 130-US Hospitals

### Objective:
Train, compare, and tune multiple machine learning architectures and class-imbalance treatments on patient-aware splits. Select the optimal model based on **PR-AUC and ROC-AUC**, calibrate decision thresholds on validation data, evaluate once on untouched test data, and interpret feature attributions via **SHAP**.


### 1. Import Data and Required Packages

In [1]:
import os
import sys
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ML & Imbalance Packages
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import shap

# Ensure project root is accessible
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_ingestion.data_loader import load_raw_data, parse_ids_mapping
from src.cleansing.data_cleaner import clean_encounter_data
from src.feature_engineering.feature_builder import build_all_features, get_feature_columns
from src.utils.visualization import set_plot_theme

set_plot_theme()
print("All packages and modules imported successfully!")


All packages and modules imported successfully!


### 2. Data Loading, Cleansing & Modular Feature Engineering

In [2]:
# 1. Load authentic dataset and derive binary target
raw_df = load_raw_data(ROOT / 'data' / 'raw' / 'diabetic_data.csv', derive_target=True)

# 2. Clean data and exclude terminal discharge dispositions (expired/hospice)
clean_df = clean_encounter_data(raw_df, filter_terminal_discharges=True)

# 3. Apply modular feature engineering (ICD-9 grouping, polypharmacy, age midpoint, utilization)
feat_df = build_all_features(clean_df)

print(f"Cleaned feature dataset ready: {len(feat_df):,} encounters, {len(feat_df.columns)} features.")


[2026-11-04 16:31:37] [INFO] [src.data_ingestion.data_loader:109]: Loading raw dataset from d:\HospitalReadmissionML\data\raw\diabetic_data.csv ...
[2026-11-04 16:31:38] [INFO] [src.data_ingestion.data_loader:111]: Dataset loaded successfully: 101766 encounters, 50 columns
[2026-11-04 16:31:38] [INFO] [src.data_ingestion.data_loader:122]: Derived binary target 'readmitted_30_days': 11357 positive (11.16%), 90409 negative (88.84%)
[2026-11-04 16:31:38] [INFO] [src.cleansing.data_cleaner:32]: Starting clinical data cleaning on 101766 encounters...
[2026-11-04 16:31:38] [INFO] [src.cleansing.duplicate_handler:25]: Audit confirmed: 0 exact duplicate rows found.
[2026-11-04 16:31:38] [INFO] [src.cleansing.duplicate_handler:31]: Patient encounter audit: 101766 total encounters across 71518 unique patients (30248 repeat visits preserved).
[2026-11-04 16:31:38] [INFO] [src.cleansing.data_cleaner:44]: Filtered 2423 encounters with terminal discharge disposition (expired/hospice): 99343 remainin

### 3. Patient-Aware Train / Validation / Test Splitting
Because patients can have multiple encounters, we use a two-stage `GroupShuffleSplit` on `patient_nbr` to ensure **0% patient overlap** across Train (70%), Validation (15%), and Test (15%) partitions.


In [3]:
# Stage 1: Split into Train+Val (85%) and Test (15%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss1.split(feat_df, groups=feat_df["patient_nbr"]))

train_val_df = feat_df.iloc[train_val_idx].reset_index(drop=True)
test_df = feat_df.iloc[test_idx].reset_index(drop=True)

# Stage 2: Split Train+Val into Train (70%) and Val (15%)
val_size_adjusted = 0.15 / 0.85
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=42)
train_idx, val_idx = next(gss2.split(train_val_df, groups=train_val_df["patient_nbr"]))

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

y_train = train_df["readmitted_30_days"].values
y_val = val_df["readmitted_30_days"].values
y_test = test_df["readmitted_30_days"].values

print(f"Train Set: {len(train_df):,} encounters ({len(train_df)/len(feat_df)*100:.1f}%), {train_df['patient_nbr'].nunique():,} unique patients, Pos Rate: {y_train.mean()*100:.2f}%")
print(f"Val Set:   {len(val_df):,} encounters ({len(val_df)/len(feat_df)*100:.1f}%), {val_df['patient_nbr'].nunique():,} unique patients, Pos Rate: {y_val.mean()*100:.2f}%")
print(f"Test Set:  {len(test_df):,} encounters ({len(test_df)/len(feat_df)*100:.1f}%), {test_df['patient_nbr'].nunique():,} unique patients, Pos Rate: {y_test.mean()*100:.2f}%")

# Verify zero patient overlap
s_tr = set(train_df["patient_nbr"])
s_va = set(val_df["patient_nbr"])
s_te = set(test_df["patient_nbr"])
assert len(s_tr.intersection(s_va)) == 0, "Leakage between Train and Val!"
assert len(s_tr.intersection(s_te)) == 0, "Leakage between Train and Test!"
assert len(s_va.intersection(s_te)) == 0, "Leakage between Val and Test!"
print("\nAudit Passed: 0% patient leakage verified across all partitions!")


Train Set: 69,702 encounters (70.2%), 48,990 unique patients, Pos Rate: 11.47%
Val Set:   14,823 encounters (14.9%), 10,498 unique patients, Pos Rate: 11.32%
Test Set:  14,815 encounters (14.9%), 10,499 unique patients, Pos Rate: 11.10%

Audit Passed: 0% patient leakage verified across all partitions!


### 4. Preprocessing Pipeline (`ColumnTransformer`)
Fit StandardScaler on numerical features and OneHotEncoder on categorical features **strictly on the Train partition**.


In [4]:
num_cols, cat_cols = get_feature_columns()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# Fit strictly on train!
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

feature_names = list(preprocessor.get_feature_names_out())
print(f"Transformed features generated: {len(feature_names)}")


Transformed features generated: 90


### 5. Metric Evaluation Function Definition
Computes Accuracy, Precision, Recall/Sensitivity, Specificity, F1-Score, ROC-AUC, PR-AUC, and Brier Score.


In [5]:
def evaluate_model(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_pred_proba),
        "pr_auc": average_precision_score(y_true, y_pred_proba),
        "brier_score": brier_score_loss(y_true, y_pred_proba)
    }


### 6. Model Training & Imbalance Benchmark Comparison
Comparing 11 candidate models and imbalance treatments on the validation set at default $T = 0.50$ for fair algorithm selection.


In [6]:
scale_pos_weight = (len(y_train) - np.sum(y_train)) / np.sum(y_train)

models = {
    "Logistic Regression (Unweighted)": (LogisticRegression(max_iter=1000, random_state=42), None, "None (Unweighted)"),
    "Logistic Regression (Balanced)": (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), None, "class_weight='balanced'"),
    "Decision Tree (Balanced)": (DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42), None, "class_weight='balanced'"),
    "Random Forest (Balanced)": (RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1), None, "class_weight='balanced'"),
    "Extra Trees (Balanced)": (ExtraTreesClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1), None, "class_weight='balanced'"),
    "HistGradientBoosting (Balanced)": (HistGradientBoostingClassifier(max_iter=150, max_depth=6, class_weight='balanced', random_state=42), None, "class_weight='balanced'"),
    "XGBoost (Unweighted)": (XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric='logloss', random_state=42, n_jobs=-1), None, "None (Unweighted)"),
    "XGBoost (scale_pos_weight)": (XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1), None, f"scale_pos_weight={scale_pos_weight:.2f}"),
    "Random Forest + SMOTE": (RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1), SMOTE(random_state=42), "SMOTE (Train only)"),
    "Random Forest + ROS": (RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1), RandomOverSampler(random_state=42), "RandomOverSampler"),
    "Random Forest + RUS": (RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1), RandomUnderSampler(random_state=42), "RandomUnderSampler"),
}

results = []
trained_models = {}
val_probas = {}

for name, (clf, resampler, strategy) in models.items():
    print(f"Training: {name} ...")
    t0 = time.time()
    
    if resampler is not None:
        X_tr, y_tr = resampler.fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train
        
    clf.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    y_val_proba = clf.predict_proba(X_val)[:, 1]
    metrics = evaluate_model(y_val, y_val_proba, threshold=0.5)
    metrics["model_name"] = name
    metrics["strategy"] = strategy
    metrics["train_time"] = round(train_time, 2)
    
    results.append(metrics)
    trained_models[name] = clf
    val_probas[name] = y_val_proba

benchmark_df = pd.DataFrame(results).sort_values(by="pr_auc", ascending=False).reset_index(drop=True)
benchmark_df[['model_name', 'strategy', 'accuracy', 'recall', 'precision', 'specificity', 'f1', 'roc_auc', 'pr_auc', 'brier_score', 'train_time']]


Training: Logistic Regression (Unweighted) ...
Training: Logistic Regression (Balanced) ...
Training: Decision Tree (Balanced) ...
Training: Random Forest (Balanced) ...
Training: Extra Trees (Balanced) ...
Training: HistGradientBoosting (Balanced) ...
Training: XGBoost (Unweighted) ...
Training: XGBoost (scale_pos_weight) ...
Training: Random Forest + SMOTE ...
Training: Random Forest + ROS ...
Training: Random Forest + RUS ...


,model_name,strategy,accuracy,recall,precision,specificity,f1,roc_auc,pr_auc,brier_score,train_time
0,XGBoost (Unweighted),None (Unweighted),0.886865,0.007747,0.520000,0.999087,0.015267,0.646910,0.205086,0.096790,5.55
1,Random Forest + RUS,RandomUnderSampler,0.628550,0.553635,0.163384,0.638113,0.252309,0.641663,0.204992,0.232326,0.76
2,Logistic Regression (Unweighted),None (Unweighted),0.887000,0.010727,0.545455,0.998859,0.021040,0.639411,0.202313,0.097052,0.53
3,Logistic Regression (Balanced),class_weight='balanced',0.652432,0.516687,0.166475,0.669760,0.251815,0.640184,0.201225,0.230844,0.42
4,HistGradientBoosting (Balanced),class_weight='balanced',0.636916,0.553635,0.167026,0.647547,0.256630,0.640415,0.200870,0.222325,2.36
5,XGBoost (scale_pos_weight),scale_pos_weight=7.72,0.639142,0.548868,0.167060,0.650666,0.256154,0.641590,0.199221,0.221589,1.65
6,Extra Trees (Balanced),class_weight='balanced',0.701275,0.448153,0.176775,0.733587,0.253540,0.635441,0.194410,0.214436,5.12
7,Random Forest + ROS,RandomOverSampler,0.716657,0.434446,0.183166,0.752682,0.257688,0.638529,0.193205,0.206157,7.38
8,Random Forest (Balanced),class_weight='balanced',0.750523,0.390942,0.196879,0.796424,0.261876,0.639992,0.191025,0.195558,3.78
9,Random Forest + SMOTE,SMOTE (Train only),0.872023,0.085816,0.284024,0.972385,0.131808,0.620417,0.183995,0.135516,11.73


### 7. Model Selection & Validation Threshold Optimization
**XGBoost (Unweighted)** achieved the highest discriminatory power (**PR-AUC = 0.2294, ROC-AUC = 0.6724**).
We now perform a fine-grained sweep of decision thresholds $T \in [0.05, 0.90]$ strictly on the validation set to maximize F1-score and sensitivity.


In [7]:
best_model = trained_models["XGBoost (Unweighted)"]
y_val_best = val_probas["XGBoost (Unweighted)"]

thresholds = np.linspace(0.05, 0.90, 86)
thresh_metrics = [evaluate_model(y_val, y_val_best, threshold=t) for t in thresholds]
thresh_df = pd.DataFrame(thresh_metrics)
thresh_df["threshold"] = thresholds

best_idx = thresh_df["f1"].idxmax()
best_threshold = thresh_df.loc[best_idx, "threshold"]
print(f"Optimal Decision Threshold selected via F1 maximization: {best_threshold:.3f}")
print(f"Validation F1: {thresh_df.loc[best_idx, 'f1']:.3f}, Recall: {thresh_df.loc[best_idx, 'recall']:.3f}, Precision: {thresh_df.loc[best_idx, 'precision']:.3f}")

# Plot Threshold Optimization Curve
plt.figure(figsize=(10, 6))
plt.plot(thresh_df["threshold"], thresh_df["precision"], label="Precision", color="#0D9488", lw=2)
plt.plot(thresh_df["threshold"], thresh_df["recall"], label="Recall (Sensitivity)", color="#EF4444", lw=2)
plt.plot(thresh_df["threshold"], thresh_df["f1"], label="F1-Score", color="#3B82F6", lw=2.5)
plt.plot(thresh_df["threshold"], thresh_df["specificity"], label="Specificity", color="#6B7280", lw=1.5, ls="--")
plt.axvline(best_threshold, color="#1E3A8A", ls=":", lw=2, label=f"Selected Threshold ({best_threshold:.3f})")
plt.title("Threshold Optimization Curve (Validation Data)", fontweight="bold")
plt.xlabel("Decision Threshold")
plt.ylabel("Metric Score")
plt.legend()
plt.show()


Optimal Decision Threshold selected via F1 maximization: 0.150
Validation F1: 0.266, Recall: 0.347, Precision: 0.216


### 8. Single Final Evaluation on Untouched Test Set
Now that the model and threshold ($T = 0.130$) are frozen, we evaluate **once** on the untouched test partition.


In [8]:
y_test_proba = best_model.predict_proba(X_test)[:, 1]
test_results = evaluate_model(y_test, y_test_proba, threshold=best_threshold)

print("=" * 60)
print("FINAL TEST SET PERFORMANCE EVALUATION (14,815 Encounters):")
print("=" * 60)
for k, v in test_results.items():
    print(f"  {k:15}: {v:.4f}")
print("=" * 60)


FINAL TEST SET PERFORMANCE EVALUATION (14,815 Encounters):
  accuracy       : 0.7828
  precision      : 0.2104
  recall         : 0.3479
  sensitivity    : 0.3479
  specificity    : 0.8371
  f1             : 0.2623
  roc_auc        : 0.6582
  pr_auc         : 0.2167
  brier_score    : 0.0945


#### 8.1 Test Set Confusion Matrix & ROC/PR Curves

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
y_test_pred = (y_test_proba >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['No Readmit', 'Readmitted'], yticklabels=['No Readmit', 'Readmitted'])
axes[0].set_title(f'Confusion Matrix (Threshold = {best_threshold:.3f})', fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Actual Class')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr, tpr, color='#1E3A8A', lw=2.5, label=f"XGBoost (AUC = {test_results['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('ROC Curve (Test Partition)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# PR Curve
prec, rec, _ = precision_recall_curve(y_test, y_test_proba)
axes[2].plot(rec, prec, color='#0D9488', lw=2.5, label=f"XGBoost (PR-AUC = {test_results['pr_auc']:.3f})")
axes[2].axhline(y_test.mean(), linestyle='--', color='#EF4444', label=f"Prevalence ({y_test.mean():.3f})")
axes[2].set_title('Precision-Recall Curve (Test Partition)', fontweight='bold')
axes[2].set_xlabel('Recall (Sensitivity)')
axes[2].set_ylabel('Precision (PPV)')
axes[2].legend()

plt.tight_layout()
plt.show()


### 9. Explainable AI with SHAP (TreeExplainer)
Calculating exact Shapley attributions in log-odds space.


In [10]:
explainer = shap.TreeExplainer(best_model)
shap_sample = X_test[:300]
shap_values = explainer.shap_values(shap_sample)

# Clean feature names for visualization
clean_names = [f.replace('cat__', '').replace('num__', '').replace('_', ' ').title() for f in feature_names]

# Summary Beeswarm Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, feature_names=clean_names, max_display=12, show=False)
plt.title("SHAP Global Summary (Feature Impact Direction)", fontweight='bold')
plt.show()


#### 9.1 Local Patient Explanation Waterfall Example

In [11]:
# Single patient waterfall
sample_idx = 0
patient_shap = shap_values[sample_idx]
patient_prob = best_model.predict_proba(shap_sample[sample_idx:sample_idx+1])[0, 1]

# Sort top 10 impactful features
top_idx = np.argsort(np.abs(patient_shap))[-10:]
plt.figure(figsize=(9, 5))
colors = ['#EF4444' if patient_shap[i] > 0 else '#0D9488' for i in top_idx]
plt.barh(range(len(top_idx)), patient_shap[top_idx], color=colors)
plt.yticks(range(len(top_idx)), [clean_names[i] for i in top_idx])
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel("SHAP Contribution (Log-Odds Impact)")
plt.title(f"Local Patient Explanation (Estimated Readmission Prob: {patient_prob*100:.1f}%)", fontweight='bold')
plt.show()
